In [0]:
df = spark.read.csv("/Volumes/workspace/default/insurance_data/insurance fraud claims.csv", header=True, inferSchema=True)
df.printSchema()
print("Row count:", df.count())

root
 |-- months_as_customer: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- policy_number: integer (nullable = true)
 |-- policy_bind_date: date (nullable = true)
 |-- policy_state: string (nullable = true)
 |-- policy_csl: string (nullable = true)
 |-- policy_deductable: integer (nullable = true)
 |-- policy_annual_premium: double (nullable = true)
 |-- umbrella_limit: integer (nullable = true)
 |-- insured_zip: integer (nullable = true)
 |-- insured_sex: string (nullable = true)
 |-- insured_education_level: string (nullable = true)
 |-- insured_occupation: string (nullable = true)
 |-- insured_hobbies: string (nullable = true)
 |-- insured_relationship: string (nullable = true)
 |-- capital-gains: integer (nullable = true)
 |-- capital-loss: integer (nullable = true)
 |-- incident_date: date (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- collision_type: string (nullable = true)
 |-- incident_severity: string (nullable = true)
 |-- authoritie

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Nulls
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+------------------+---+-------------+----------------+------------+----------+-----------------+---------------------+--------------+-----------+-----------+-----------------------+------------------+---------------+--------------------+-------------+------------+-------------+-------------+--------------+-----------------+---------------------+--------------+-------------+-----------------+------------------------+---------------------------+---------------+---------------+---------+-----------------------+------------------+------------+--------------+-------------+---------+----------+---------+--------------+----+
|months_as_customer|age|policy_number|policy_bind_date|policy_state|policy_csl|policy_deductable|policy_annual_premium|umbrella_limit|insured_zip|insured_sex|insured_education_level|insured_occupation|insured_hobbies|insured_relationship|capital-gains|capital-loss|incident_date|incident_type|collision_type|incident_severity|authorities_contacted|incident_state|incident_c

In [0]:
# '?' placeholder values
for c in ["collision_type", "property_damage", "police_report_available"]:
    print(c, df.filter(col(c) == "?").count())

collision_type 178
property_damage 360
police_report_available 343


In [0]:
df = spark.read.csv("/Volumes/workspace/default/insurance_data/insurance fraud claims.csv", header=True, inferSchema=True)
print("Row count:", df.count())

Row count: 1000


In [0]:
df_clean = df.drop("_c39")

In [0]:
from pyspark.sql.functions import when, col

for c in ["collision_type", "property_damage", "police_report_available"]:
    df_clean = df_clean.withColumn(c, when(col(c) == "?", "Unknown").otherwise(col(c)))

In [0]:
df_clean = df_clean.na.fill({"authorities_contacted": "None"})

In [0]:
from pyspark.sql.functions import round as spark_round

df_clean = df_clean.withColumn("claim_to_premium_ratio",
    spark_round(col("total_claim_amount") / col("policy_annual_premium"), 2))

df_clean = df_clean.withColumn("high_risk_flag",
    when((col("incident_severity") == "Major Damage") &
         (col("police_report_available") == "Unknown") &
         (col("total_claim_amount") > 50000), "High Risk")
    .otherwise("Low Risk"))

In [0]:
assert df_clean.filter(col("policy_number").isNull()).count() == 0
assert df_clean.filter(col("total_claim_amount") < 0).count() == 0
assert "_c39" not in df_clean.columns
print("Validation passed. Final row count:", df_clean.count())

Validation passed. Final row count: 1000


In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("insurance_claims_clean")

In [0]:
%sql
SELECT incident_severity, 
       ROUND(SUM(CASE WHEN fraud_reported = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS fraud_rate_pct,
       COUNT(*) AS total_claims
FROM insurance_claims_clean
GROUP BY incident_severity
ORDER BY fraud_rate_pct DESC

incident_severity,fraud_rate_pct,total_claims
Major Damage,60.5,276
Total Loss,12.9,280
Minor Damage,10.7,354
Trivial Damage,6.7,90


In [0]:
%sql
SELECT fraud_reported, ROUND(AVG(total_claim_amount),2) AS avg_claim_amount
FROM insurance_claims_clean
GROUP BY fraud_reported

fraud_reported,avg_claim_amount
Y,60302.11
N,50288.61


In [0]:
%sql
SELECT collision_type,
       ROUND(SUM(CASE WHEN fraud_reported = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS fraud_rate_pct,
       COUNT(*) AS total_claims
FROM insurance_claims_clean
GROUP BY collision_type
ORDER BY fraud_rate_pct DESC

collision_type,fraud_rate_pct,total_claims
Rear Collision,31.2,292
Front Collision,27.6,254
Side Collision,25.4,276
Unknown,9.0,178


In [0]:
%sql
SELECT police_report_available,
       ROUND(SUM(CASE WHEN fraud_reported = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS fraud_rate_pct,
       COUNT(*) AS total_claims
FROM insurance_claims_clean
GROUP BY police_report_available
ORDER BY fraud_rate_pct DESC

police_report_available,fraud_rate_pct,total_claims
Unknown,25.9,343
NO,25.1,343
YES,22.9,314
